# Name That LLM | Who am I?

In [ ]:
import gc, glob, json, os, re, shutil, time
SOURCE_CREDIT = "Based on Nishant Dahal, Two-stage LoRA part two: https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool; public artifacts: https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend"


os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn


torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"


W_SVC = 0.75
LORAS = {
    "lora_qwen3_14b":  dict(w=0.55, twin_w=0.15, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.25, twin_w=0.10, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.30
TWIN = dict(thresh=0.85, min_size=2, max_size=18)


LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    lp = np.log(np.clip(P, 1e-9, 1))
    LOGIT[rows] += w * lp
    cache = np.zeros(LOGIT.shape, dtype=np.float32)
    cache[rows] = lp.astype(np.float32)
    MEMBER_LOGP[name] = cache
    MEMBER_WEIGHTS[name] = float(w)
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


TRAIN_PATH = "/kaggle/input/competitions/name-that-llm/train.csv"
TEST_PATH = "/kaggle/input/competitions/name-that-llm/test.csv"
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = find("train.csv")
if not os.path.exists(TEST_PATH):
    TEST_PATH = find("test.csv")
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
test["text"] = test["text"].fillna("").astype(str)
assert test["id"].notna().all() and test["id"].is_unique
assert train["model"].isin(LABELS).all()
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
MEMBER_LOGP = {}
MEMBER_WEIGHTS = {}
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

PUBLIC_SAMPLE = len(test) < 50
if PUBLIC_SAMPLE:
    fallback = train["model"].mode().iat[0]
    pd.DataFrame({"id": test["id"].astype(str), "model": fallback}).to_parquet("submission.parquet", index=False)
    print("fast public validation output written; hidden scoring runs the full ensemble", flush=True)



In [ ]:

OURS = dict(modernbert=0.30, desklib=0.05)
TYPO_GUARDS = False


In [ ]:
if not PUBLIC_SAMPLE:
    import string
    from scipy.sparse import hstack
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.svm import LinearSVC

    SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    VIEWS = [
        ("char", 3, 5, 250000, 1.0),
        ("word", 1, 3, 180000, 0.7),
        ("shape", 2, 6, 60000, 0.5),
        ("small", 1, 3, 50000, 0.5),
        ("punct", 2, 4, 30000, 0.5)
    ]

    def shape_text(t):
        """Every run of letters becomes X or x, so only the rhythm is left."""
        return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", str(t))

    def small_words_text(t):
        """Keep the small words and the punctuation, and replace every other word by CONTENT."""
        return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                        for w in re.findall(r"\w+|[^\w\s]", str(t)))

    PUNCT_SET = set(string.punctuation).union({chr(0x2014), chr(0x2013), chr(0x2018), chr(0x2019), chr(0x201c), chr(0x201d)})

    def punct_text(t):
        """Extract punctuation rhythm: dashes, quotes, colons, brackets, and surrounding spacing."""
        return "".join(c for c in str(t) if c.isspace() or c in PUNCT_SET)

    pool = pd.read_parquet(find("pool_v4.parquet"))
    fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
    fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]

    try:
        ext_csv = find("passages_clean.csv")
        ext_df = pd.read_csv(ext_csv)
        ext_df = ext_df[ext_df["model"].isin(LABELS)]
        fit_texts += ext_df["text"].astype(str).tolist()
        fit_y = np.r_[fit_y, ext_df["model"].map({l: i for i, l in enumerate(LABELS)}).values]
        print(f"Attached {len(ext_df)} curated passages from passages_clean.csv", flush=True)
    except Exception as e:
        print(f"passages_clean.csv note: {e}", flush=True)

    A, B = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        if view == "punct":
            prep = punct_text
            analyzer = "char"
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32)
        elif view == "shape":
            prep = shape_text
            analyzer = "char"
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32)
        elif view == "small":
            prep = small_words_text
            analyzer = "word"
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32, token_pattern=r"(?u)\w+|[^\w\s]")
        elif view == "word":
            prep = lambda s: str(s)
            analyzer = "word"
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32, token_pattern=r"(?u)\w+|[^\w\s]")
        else:
            prep = lambda s: str(s)
            analyzer = "char"
            opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                        max_features=max_features, dtype=np.float32)
        vec = TfidfVectorizer(**opts)
        A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
        B.append(vec.transform([prep(t) for t in test.text]) * weight)

    svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
    SVC_RAW = svc.decision_function(hstack(B, format="csr")).astype(np.float32)
    # Calibrated temperature scaling (1.25)
    SVC_P = softmax(SVC_RAW / 1.25)
    vote("svc", SVC_P, W_SVC)
    del A, B, svc, pool
    gc.collect()


In [ ]:
if not PUBLIC_SAMPLE:
    from safetensors.torch import load_file
    from transformers import AutoConfig, AutoModel, AutoTokenizer


    def find_base(spec):
        """The attached base model whose config.json matches spec."""
        for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if all(c.get(k) == v for k, v in spec.items()):
                return os.path.dirname(cfg)
        raise FileNotFoundError(f"attach the base model {spec}")


    def load_trunk(base_dir, keep, dtype):
        """The first keep decoder layers, split over both T4s when they do not fit on one."""
        mc = AutoConfig.from_pretrained(base_dir)
        mc.num_hidden_layers, mc.use_cache = keep, False
        if getattr(mc, "layer_types", None) is not None:
            mc.layer_types = mc.layer_types[:keep]
        H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
        hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
        q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
        gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
        if gib > 13 and torch.cuda.device_count() > 1:
            mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
        else:
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
        model.norm = nn.Identity()
        print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
        return model


    @torch.no_grad()
    def merge_lora(model, adapter_dir):
        """W becomes W + B A alpha / r for every adapted linear layer."""
        acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
        scale = acfg["lora_alpha"] / acfg["r"]
        sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
        modules = dict(model.named_modules())
        for k, A in sd.items():
            if ".lora_A." not in k:
                continue
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
            lin.weight.add_(delta.to(lin.weight.dtype))


    @torch.no_grad()
    def lora_features(ckpt_dir, base_dir, texts, bs):
        """Pooled features Z, one row per passage, and the checkpoint's heads."""
        meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
        assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
        cfg = meta["cfg"]
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        tok = AutoTokenizer.from_pretrained(base_dir)
        tok.padding_side = "right"
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = load_trunk(base_dir, cfg["keep"], dtype)
        merge_lora(model, os.path.join(ckpt_dir, "adapter"))
        heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
        layers, H = cfg["layers"], model.config.hidden_size
        norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
        norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
        ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]

        eos_id = cfg.get("eos_id", tok.eos_token_id)
        print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
        ids = [x + [eos_id] for x in ids]
        order = np.argsort([len(x) for x in ids], kind="stable")
        Z = np.zeros((len(ids), H * len(layers)), np.float32)

        def embed(idx):
            enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in layers):
                model.to(torch.bfloat16)
                hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
            m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
            m[:, 0] = 0
            den = m.sum(1).clamp(min=1)
            Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            try:
                embed(idx)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                for i in idx:
                    embed([i])
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

In [ ]:
if not PUBLIC_SAMPLE:
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    if len(test) >= 2:
        Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
        D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
        np.fill_diagonal(D, 0)
        TREE = linkage(squareform(D, checks=False), method="average")
        groups_tight = fcluster(TREE, 0.82, criterion="distance")
        groups_broad = fcluster(TREE, 0.88, criterion="distance")
        del D
    else:
        groups_tight = np.ones(len(test), dtype=int)
        groups_broad = np.ones(len(test), dtype=int)
        TREE = None

    size_tight = np.bincount(groups_tight)[groups_tight]
    size_broad = np.bincount(groups_broad)[groups_broad]
    twin_rows = (size_broad >= 2) & (size_broad <= 18)
    print(f"Multi-scale topic clusters: {int((size_broad == 1).sum())} alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

    for ckpt_name, mc in LORAS.items():
        ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
        Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
        vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
        if twin_rows.any():
            Zc = Z.copy()
            # Multi-scale mean subtraction: 70% tight prompt cluster + 30% broad topic cluster
            for g in np.unique(groups_broad[twin_rows]):
                m_broad = Zc[groups_broad == g].mean(0)
                Zc[groups_broad == g] -= 0.30 * m_broad
            for g in np.unique(groups_tight[twin_rows]):
                m_tight = Zc[groups_tight == g].mean(0)
                Zc[groups_tight == g] -= 0.70 * m_tight
            centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
            vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
        del Z
        gc.collect()


In [ ]:
if not PUBLIC_SAMPLE:
    ck = os.path.dirname(find("deberta_v3_large/meta.json"))
    if ck.startswith("/kaggle/input"):
        shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
        ck = "/tmp/deberta_v3_large"
    meta = json.load(open(os.path.join(ck, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
    etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
    enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
    head = load_file(os.path.join(ck, "head.safetensors"))
    W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
    ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for s in range(0, len(order), 32):
            idx = order[s:s + 32]
            bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
            m = bt["attention_mask"].unsqueeze(-1).float()
            Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
    del enc
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
%%writefile enc_infer.py
"""One encoder on one T4: fp32 weights + fp16 autocast (T4 tensor cores), length-sorted dynamic padding,
fp32 retry for any batch that overflows in fp16. Writes raw logits + the head's label order."""
import sys, json, os, time, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
path, data, out = sys.argv[1:4]; t0 = time.time()
test = pd.read_csv(data); texts = test.text.fillna('').astype(str).tolist()
tok = AutoTokenizer.from_pretrained(path)
model = AutoModelForSequenceClassification.from_pretrained(path, dtype=torch.float32).cuda().eval()
labels = [model.config.id2label[i] for i in range(model.config.num_labels)]
ids = tok(texts, truncation=True, max_length=512)['input_ids']; order = np.argsort([len(x) for x in ids])
logits = np.zeros((len(texts), len(labels)), np.float32); retried = 0
with torch.inference_mode():
    for s in range(0, len(order), 64):
        idx = order[s:s + 64]; b = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to('cuda')
        with torch.autocast('cuda', dtype=torch.float16):
            lg = model(**b).logits.float()
        if not torch.isfinite(lg).all():
            lg = model(**b).logits.float(); retried += 1
        logits[idx] = lg.cpu().numpy()
assert np.isfinite(logits).all()
np.savez(out, logits=logits, labels=np.array(labels, dtype=str), ids=test['id'].astype(str).to_numpy(dtype=str))   # unicode, not object
print(f'{os.path.basename(path)}: {len(texts)} rows in {time.time() - t0:.0f}s, fp32 retries {retried}', flush=True)

In [ ]:
if not PUBLIC_SAMPLE:

    import subprocess, sys
    torch.cuda.empty_cache()
    try:
        OURS_ROOT = os.path.dirname(find("light_members.npz"))
    except FileNotFoundError:
        OURS_ROOT = None
        print("WARNING: ModernBERT / Desklib dataset not attached. Expected public input nuridudak/ntl-ours-v1.")
    jobs = {}
    for gpu, name in enumerate([n for n, w in OURS.items() if w > 0 and OURS_ROOT]):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu % max(1, torch.cuda.device_count())))
        jobs[name] = subprocess.Popen([sys.executable, "enc_infer.py", os.path.join(OURS_ROOT, name), TEST_PATH, f"/kaggle/working/{name}_logits.npz"],
                                      env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for name, job in jobs.items():
        out, _ = job.communicate()
        print("\n".join(l for l in out.splitlines() if "arn" not in l)[-1500:], flush=True)
        assert job.returncode == 0, f"{name} failed with exit code {job.returncode}: {out[-2500:]}"
        z = np.load(f"/kaggle/working/{name}_logits.npz")
        assert list(z["ids"]) == test["id"].astype(str).tolist(), "row order differs"
        lab = [str(l) for l in z["labels"]]
        Pm = softmax(z["logits"][:, [lab.index(l) for l in LABELS]].astype(np.float64))
        assert np.isfinite(Pm).all()
        vote(name, Pm, OURS[name])

In [ ]:
def grouped_svc_shift(scores, group_ids, min_size=8, max_size=18):
    shift = np.zeros_like(scores, dtype=np.float64)
    counts = np.bincount(group_ids)[group_ids]
    valid = (counts >= min_size) & (counts <= max_size)
    for gid in np.unique(group_ids[valid]):
        idx = np.flatnonzero(group_ids == gid)
        center = np.mean(scores[idx], axis=0, dtype=np.float64)
        shift[idx] = center.mean() - center
    return shift, valid


if not PUBLIC_SAMPLE:
    def sinkhorn(P, iters=300):
        c = np.ones(K, dtype=np.float64)
        target = np.full(K, len(P) / K)
        for _ in range(iters):
            R = P * c
            R /= R.sum(axis=1, keepdims=True)
            mass = R.sum(axis=0)
            c *= target / np.maximum(mass, 1e-12)
        R = P * c
        return R / R.sum(axis=1, keepdims=True)

    def save_prediction(scores, filename):
        P = sinkhorn(softmax(scores))
        pred = np.asarray(LABELS)[P.argmax(axis=1)]
        sub = pd.DataFrame({'id': test.id.astype(str), 'model': pred})
        assert len(sub) == len(test) and sub.id.is_unique
        assert sub.model.isin(LABELS).all() and sub.id.notna().all()
        sub.to_parquet(filename, index=False)
        return pred

    original_scores = np.asarray(LOGIT, dtype=np.float64).copy()

    # --- Soft Typographic Prior Gating ---
    row_margin = np.sort(original_scores, axis=1)[:, -1] - np.sort(original_scores, axis=1)[:, -2]
    uncertainty = np.clip((2.0 - row_margin) / 2.0, 0.0, 1.0)

    curly_apos = chr(0x2019)
    curly_open_q = chr(0x201c)
    curly_close_q = chr(0x201d)
    em_dash = chr(0x2014)

    claude_glm_idx = [i for i, l in enumerate(LABELS) if l.startswith(("claude", "glm"))]
    gpt_idx = [i for i, l in enumerate(LABELS) if l.startswith("gpt")]

    typo_adjust = np.zeros_like(original_scores)
    for i, t in enumerate(test.text.values):
        s_text = str(t)
        u = uncertainty[i]
        if u > 0.05:
            if (curly_apos in s_text or curly_open_q in s_text or curly_close_q in s_text):
                typo_adjust[i, claude_glm_idx] -= 0.8 * u
            if (em_dash in s_text or " — " in s_text or " -- " in s_text):
                typo_adjust[i, claude_glm_idx] += 0.5 * u

    adjusted_scores = original_scores + typo_adjust

    topic_groups = fcluster(TREE, 0.90, criterion='distance')
    shift, accepted = grouped_svc_shift(SVC_RAW, topic_groups, 8, 18)
    gate = np.clip((1.9 - row_margin) / 1.5, 0.12, 1.0)
    gate[~accepted] = 0.0
    delta = W_SVC * shift * gate[:, None]

    # V5: Calibrated 3-Point Convex Mixture around the proven sweet spot (0.50 - 0.72)
    P_c50 = sinkhorn(softmax(adjusted_scores + 0.50 * delta))
    P_c62 = sinkhorn(softmax(adjusted_scores + 0.62 * delta))
    P_c72 = sinkhorn(softmax(adjusted_scores + 0.72 * delta))
    P_final = 0.30 * P_c50 + 0.50 * P_c62 + 0.20 * P_c72
    pred_final = np.asarray(LABELS)[P_final.argmax(axis=1)]

    sub = pd.DataFrame({'id': test.id.astype(str), 'model': pred_final})
    assert len(sub) == len(test) and sub.id.is_unique
    assert sub.model.isin(LABELS).all() and sub.id.notna().all()
    sub.to_parquet('submission.parquet', index=False)
    print('V5 Multi-Scale Centred Twin + Typo-Regularized submission.parquet written successfully!', flush=True)

    cache = dict(
        ids=test.id.astype(str).to_numpy(dtype=str),
        labels=np.asarray(LABELS, dtype=str),
        logit=adjusted_scores.astype(np.float32),
        svc_raw=SVC_RAW.astype(np.float32),
        topic_groups=np.asarray(topic_groups, dtype=np.int32),
        group_mask=np.asarray(accepted, dtype=np.uint8),
        gate=np.asarray(gate, dtype=np.float32),
        member_info=np.asarray(json.dumps([
            {'key': f'member_{i}', 'name': name, 'weight': MEMBER_WEIGHTS[name]}
            for i, name in enumerate(MEMBER_LOGP)
        ]))
    )
    for i, name in enumerate(MEMBER_LOGP):
        cache[f'member_{i}'] = MEMBER_LOGP[name]
    np.savez_compressed('v5_inference_cache.npz', **cache)

    print('complete topic groups', int(accepted.sum()), '/', len(test))
    print('group sizes', pd.Series(np.bincount(topic_groups)[topic_groups]).value_counts().sort_index().to_dict())
    print('loaded', list(MEMBER_LOGP), 'elapsed', round(time.time() - T0, 1), 'seconds')
